In [0]:
# =============================================================================
# NOTEBOOK  : 04_create_dim_date
# PURPOSE   : Generate gold.dim_date — calendar dimension table
# LAYER     : Gold (dimension table)
# RUN       : Once — no refresh needed unless date range needs extending
# =============================================================================

import sys, json
from datetime import date

_notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
PROJECT_ROOT = "/Workspace" + _notebook_path.split("/mini_project_a3t7/")[0] + "/mini_project_a3t7"

with open(f"{PROJECT_ROOT}/config/config.json") as f:
    cfg = json.load(f)

# ── Generate date range ───────────────────────────────────────────────────────
# Adjust start_date and end_date as needed for your data range
spark.sql("""
    CREATE OR REPLACE TABLE azure3_team7_project.gold.dim_date
    USING DELTA
    TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite' = 'true',
        'delta.autoOptimize.autoCompact'   = 'true'
    )
    COMMENT 'Gold: Calendar dimension. One row per date. Generated once.'
    AS
    WITH date_spine AS (
        SELECT explode(sequence(
            to_date('2020-01-01'),
            to_date('2030-12-31'),
            interval 1 day
        )) AS date
    )
    SELECT
        date                                            AS date,
        CAST(date_format(date, 'yyyyMMdd') AS INT)      AS date_key,
        year(date)                                      AS year,
        quarter(date)                                   AS quarter,
        month(date)                                     AS month,
        day(date)                                       AS day,
        dayofweek(date)                                 AS day_of_week,      -- 1=Sunday, 7=Saturday
        dayofyear(date)                                 AS day_of_year,
        weekofyear(date)                                AS week_of_year,
        date_format(date, 'MMMM')                       AS month_name,       -- January, February...
        date_format(date, 'MMM')                        AS month_short,      -- Jan, Feb...
        date_format(date, 'EEEE')                       AS day_name,         -- Monday, Tuesday...
        date_format(date, 'EEE')                        AS day_short,        -- Mon, Tue...
        date_format(date, 'yyyy-MM')                    AS year_month,       -- 2024-01
        date_format(date, 'yyyy-Qq')                    AS year_quarter,     -- 2024-Q1
        CASE WHEN dayofweek(date) IN (1, 7) THEN true
             ELSE false END                             AS is_weekend,
        CASE WHEN dayofweek(date) IN (1, 7) THEN false
             ELSE true END                             AS is_weekday,
        CASE WHEN month(date) IN (12, 1, 2) THEN 'Winter'
             WHEN month(date) IN (3, 4, 5)  THEN 'Spring'
             WHEN month(date) IN (6, 7, 8)  THEN 'Summer'
             ELSE 'Autumn' END                          AS season,
        CAST(date_format(date, 'yyyyMM') AS INT)        AS year_month_key    -- 202401
    FROM date_spine
    ORDER BY date
""")

print("[DONE] gold.dim_date created")
print("Row count:", spark.read.table("azure3_team7_project.gold.dim_date").count())